# 🚀 ECommerce-IA — Entraînement EfficientNet-B4 sur Google Colab

**Projet SAHELYS** — Fine-tuning EfficientNet-B4 pour la classification de 120 catégories de produits e-commerce.

| Paramètre | Valeur |
|---|---|
| Modèle | EfficientNet-B4 (timm) — 19.3M params |
| Dataset | Fashion Product Images (120 catégories, 720 images) |
| GPU | T4 16 Go (Colab free tier) |
| Epochs | 30 (early stopping patience=7) |
| Image size | 380×380 |
| Batch | 16 (optimisé GPU) |
| Optimizer | AdamW + CosineAnnealingLR |
| Loss | CrossEntropy + Label Smoothing 0.1 |
| Stratégie | Unfreeze progressif (3 phases) |
| Augmentation | RandomHorizontalFlip + ColorJitter + RandomRotation |

**Auteur** : BAWANA Théodore

---

### ⚠️ Avant de commencer
1. Menu **Exécution** → **Modifier le type d'exécution** → **GPU T4**
2. Cliquez **Exécuter tout** (Ctrl+F9)
3. À la fin, le modèle sera téléchargé automatiquement (~80 Mo)

## 1️⃣ Vérification GPU & Installation

In [ ]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} Go")
else:
    print("\n⚠️  AUCUN GPU détecté !")
    print("   Menu Exécution → Modifier le type d'exécution → GPU T4")

In [ ]:
!pip install -q timm kaggle tqdm pillow

## 2️⃣ Téléchargement du Dataset depuis Kaggle

In [ ]:
import os
import json

# === CONFIGURATION KAGGLE ===
# Option A : Mettre vos credentials ici
KAGGLE_USERNAME = "tedrostovenim"
KAGGLE_KEY = ""  # <-- Collez votre clé Kaggle ici

# Option B : Uploader kaggle.json via l'icône fichier (à gauche)
# Placez kaggle.json dans /root/.kaggle/

if KAGGLE_KEY:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print(f"✅ Kaggle configuré pour : {KAGGLE_USERNAME}")
else:
    # Vérifier si kaggle.json existe déjà
    kaggle_path = os.path.expanduser("~/.kaggle/kaggle.json")
    if os.path.exists(kaggle_path):
        print("✅ kaggle.json déjà présent")
    else:
        print("⚠️  Veuillez remplir KAGGLE_KEY ci-dessus ou uploader kaggle.json")
        print("   Allez sur https://www.kaggle.com/settings → API → Create New Token")

In [ ]:
%%time
import os

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Télécharger le dataset Fashion Product Images (petit, ~565 Mo)
print("📥 Téléchargement du dataset...")
!kaggle datasets download -d paramaggarwal/fashion-product-images-small -p {DATA_DIR} --unzip

# Télécharger aussi styles.csv (métadonnées) depuis le dataset complet
print("\n📥 Téléchargement des métadonnées...")
!kaggle datasets download -d paramaggarwal/fashion-product-images-dataset -f styles.csv -p {DATA_DIR}

# Décompresser styles.csv s'il est zippé
import zipfile
styles_zip = os.path.join(DATA_DIR, "styles.csv.zip")
if os.path.exists(styles_zip):
    with zipfile.ZipFile(styles_zip, 'r') as z:
        z.extractall(DATA_DIR)
    os.remove(styles_zip)

# Vérifier
print("\n📁 Fichiers téléchargés :")
for item in os.listdir(DATA_DIR):
    path = os.path.join(DATA_DIR, item)
    if os.path.isdir(path):
        count = len(os.listdir(path))
        print(f"   📂 {item}/ ({count} fichiers)")
    else:
        size_mb = os.path.getsize(path) / 1e6
        print(f"   📄 {item} ({size_mb:.1f} Mo)")

## 3️⃣ Organisation des données (120 catégories × 6 images)

In [ ]:
import pandas as pd
import shutil
from pathlib import Path
import random
import numpy as np

random.seed(42)
np.random.seed(42)

DATA_DIR = Path("/content/data")
IMAGES_DIR = DATA_DIR / "images"  # fashion-product-images-small
RAW_DIR = DATA_DIR / "raw"
SPLITS_DIR = DATA_DIR / "splits"

# Charger les métadonnées
styles_path = DATA_DIR / "styles.csv"
print(f"📥 Chargement de {styles_path}...")
df = pd.read_csv(styles_path, on_bad_lines='skip')
print(f"   {len(df)} produits chargés")
print(f"   Colonnes : {list(df.columns)}")

# Trouver le dossier d'images
if not IMAGES_DIR.exists():
    # Chercher le dossier contenant les images
    for d in DATA_DIR.iterdir():
        if d.is_dir() and len(list(d.glob("*.jpg"))) > 100:
            IMAGES_DIR = d
            break
    # Peut être dans myntradataset/images
    for d in DATA_DIR.rglob("images"):
        if d.is_dir() and len(list(d.glob("*.jpg"))) > 100:
            IMAGES_DIR = d
            break

nb_images = len(list(IMAGES_DIR.glob("*.jpg")))
print(f"\n🖼️  {nb_images} images trouvées dans {IMAGES_DIR}")

In [ ]:
# === Sélection de 120 catégories avec au moins 6 images ===

NB_CATEGORIES = 120
IMAGES_PAR_CAT = 6

# Vérifier quelles images existent réellement
existing_ids = set()
for img_file in IMAGES_DIR.glob("*.jpg"):
    try:
        existing_ids.add(int(img_file.stem))
    except ValueError:
        pass

df_valid = df[df['id'].isin(existing_ids)].copy()
print(f"✅ {len(df_valid)} produits avec images valides")

# Compter par catégorie
cat_counts = df_valid['articleType'].value_counts()
eligible = cat_counts[cat_counts >= IMAGES_PAR_CAT]
print(f"✅ {len(eligible)} catégories avec >= {IMAGES_PAR_CAT} images")

# Sélectionner les 120 premières
selected_categories = eligible.head(NB_CATEGORIES).index.tolist()
print(f"✅ {len(selected_categories)} catégories sélectionnées")

# Sélectionner 6 images aléatoires par catégorie
selected_products = []
for cat in selected_categories:
    cat_products = df_valid[df_valid['articleType'] == cat]
    sampled = cat_products.sample(n=IMAGES_PAR_CAT, random_state=42)
    selected_products.append(sampled)

df_selected = pd.concat(selected_products, ignore_index=True)
print(f"\n📦 Total : {len(df_selected)} images sélectionnées ({len(selected_categories)} catégories × {IMAGES_PAR_CAT})")

In [ ]:
# === Organiser en dossiers par catégorie ===

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True)

copied = 0
errors = 0

for _, row in df_selected.iterrows():
    cat = str(row['articleType']).strip()
    img_id = row['id']
    src = IMAGES_DIR / f"{img_id}.jpg"
    
    if src.exists():
        cat_dir = RAW_DIR / cat
        cat_dir.mkdir(exist_ok=True)
        dst = cat_dir / f"{img_id}.jpg"
        shutil.copy2(src, dst)
        copied += 1
    else:
        errors += 1

print(f"✅ {copied} images copiées dans {RAW_DIR}")
if errors:
    print(f"⚠️  {errors} images non trouvées")

# Vérifier
cats = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
print(f"📂 {len(cats)} dossiers créés")

In [ ]:
# === Répartition en Train/Val/Test (4/1/1 par catégorie) ===

if SPLITS_DIR.exists():
    shutil.rmtree(SPLITS_DIR)

for split in ['train', 'val', 'test']:
    (SPLITS_DIR / split).mkdir(parents=True)

split_stats = {'train': 0, 'val': 0, 'test': 0}

for cat_dir in sorted(RAW_DIR.iterdir()):
    if not cat_dir.is_dir():
        continue
    
    images = sorted(list(cat_dir.glob("*.jpg")))
    random.shuffle(images)
    
    # 6 images : 4 train, 1 val, 1 test
    n = len(images)
    n_test = max(1, n // 6)
    n_val = max(1, n // 6)
    n_train = n - n_val - n_test
    
    splits = {
        'train': images[:n_train],
        'val': images[n_train:n_train + n_val],
        'test': images[n_train + n_val:]
    }
    
    for split_name, split_images in splits.items():
        dest_dir = SPLITS_DIR / split_name / cat_dir.name
        dest_dir.mkdir(parents=True, exist_ok=True)
        for img in split_images:
            shutil.copy2(img, dest_dir / img.name)
            split_stats[split_name] += 1

print("✅ Répartition terminée :")
for split, count in split_stats.items():
    n_cats = len(list((SPLITS_DIR / split).iterdir()))
    print(f"   {split:5s} : {count} images ({n_cats} catégories)")

# Sauvegarder le class_mapping
class_mapping = {cat: idx for idx, cat in enumerate(sorted(cats))}
with open(DATA_DIR / "class_mapping.json", "w") as f:
    json.dump(class_mapping, f, indent=2)
print(f"\n💾 class_mapping.json sauvé ({len(class_mapping)} classes)")

## 4️⃣ Dataset PyTorch & DataLoaders

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import timm
import numpy as np
import json
import time
import os

# ============================================
# Constantes
# ============================================
IMAGE_SIZE = 380
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("/content/data")
SPLITS_DIR = DATA_DIR / "splits"
MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"🖥️  Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU : {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================
# Dataset PyTorch
# ============================================
class ProductDataset(Dataset):
    """Dataset pour la classification de produits e-commerce."""
    
    def __init__(self, root_dir, transform=None, class_mapping=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.extensions = {".jpg", ".jpeg", ".png", ".webp"}
        
        if class_mapping:
            self.class_to_idx = class_mapping
        else:
            self.class_to_idx = {
                d.name: idx for idx, d in enumerate(
                    sorted([d for d in self.root_dir.iterdir() if d.is_dir()])
                )
            }
        
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.num_classes = len(self.class_to_idx)
        
        # Charger les images
        self.images, self.labels = [], []
        for cat_name, cat_idx in self.class_to_idx.items():
            cat_dir = self.root_dir / cat_name
            if not cat_dir.exists():
                continue
            for img_path in sorted(cat_dir.iterdir()):
                if img_path.suffix.lower() in self.extensions:
                    self.images.append(img_path)
                    self.labels.append(cat_idx)
        
        print(f"   📦 {len(self)} images, {self.num_classes} classes depuis {self.root_dir.name}/")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        return image, label


# ============================================
# Transformations avec augmentation pour Train
# ============================================
train_transform = T.Compose([
    T.Resize((IMAGE_SIZE + 20, IMAGE_SIZE + 20)),
    T.RandomCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    T.RandomErasing(p=0.1),
])

val_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Charger le class_mapping
with open(DATA_DIR / "class_mapping.json") as f:
    class_mapping = json.load(f)

# Créer les datasets
print("\n📦 Chargement des datasets :")
train_dataset = ProductDataset(SPLITS_DIR / "train", train_transform, class_mapping)
val_dataset = ProductDataset(SPLITS_DIR / "val", val_transform, class_mapping)
test_dataset = ProductDataset(SPLITS_DIR / "test", val_transform, class_mapping)

NUM_CLASSES = train_dataset.num_classes

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n✅ DataLoaders prêts :")
print(f"   Train : {len(train_dataset)} images → {len(train_loader)} batches")
print(f"   Val   : {len(val_dataset)} images → {len(val_loader)} batches")
print(f"   Test  : {len(test_dataset)} images → {len(test_loader)} batches")
print(f"   Classes : {NUM_CLASSES}")

## 5️⃣ Modèle EfficientNet-B4

In [ ]:
# ============================================
# Création du modèle EfficientNet-B4
# ============================================
DROPOUT_RATE = 0.3

print("🏗️  Création du modèle EfficientNet-B4...")
model = timm.create_model(
    "efficientnet_b4",
    pretrained=True,
    num_classes=NUM_CLASSES,
    drop_rate=DROPOUT_RATE
)

# Tête personnalisée
in_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT_RATE),
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"   Paramètres totaux : {total_params:,}")
print(f"   Classes         : {NUM_CLASSES}")
print(f"   Input size      : {IMAGE_SIZE}×{IMAGE_SIZE}")

## 6️⃣ Fonctions d'entraînement

In [ ]:
# ============================================
# Unfreeze Progressif
# ============================================
def geler_backbone(model):
    """Phase 1 : seule la tête est entraînée."""
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   ❄️  Backbone gelé — {n:,} params entraînables")

def degeler_derniers_blocs(model, nb_blocs=2):
    """Phase 2 : derniers blocs + tête."""
    blocks = list(model.blocks)
    for block in blocks[-nb_blocs:]:
        for param in block.parameters():
            param.requires_grad = True
    for name, param in model.named_parameters():
        if "conv_head" in name or "bn2" in name:
            param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🔓 Derniers {nb_blocs} blocs dégelés — {n:,} params entraînables")

def degeler_tout(model):
    """Phase 3 : tout le réseau."""
    for param in model.parameters():
        param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🔓 Réseau entièrement dégelé — {n:,} params entraînables")


# ============================================
# Label Smoothing CrossEntropy
# ============================================
class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing
    
    def forward(self, pred, target):
        log_probs = torch.nn.functional.log_softmax(pred, dim=-1)
        nll = -log_probs.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth = -log_probs.mean(dim=-1)
        return (self.confidence * nll + self.smoothing * smooth).mean()


# ============================================
# Early Stopping
# ============================================
class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.should_stop = False
    
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"   ⏹️ Early stopping (patience={self.patience})")
        return self.should_stop


# ============================================
# Train / Validate
# ============================================
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        loss_sum += loss.item() * images.size(0)
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100.*correct/total:.1f}%")
    return loss_sum / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    loss_sum, correct1, correct5, total = 0.0, 0, 0, 0
    for images, labels in tqdm(loader, desc="  Val  ", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss_sum += loss.item() * images.size(0)
        total += labels.size(0)
        _, p1 = outputs.max(1)
        correct1 += p1.eq(labels).sum().item()
        _, p5 = outputs.topk(min(5, outputs.size(1)), dim=1)
        correct5 += sum(labels[i].item() in p5[i].tolist() for i in range(labels.size(0)))
    return loss_sum / total, correct1 / total, correct5 / total

print("✅ Fonctions d'entraînement définies")

## 7️⃣ Entraînement (30 epochs, unfreeze progressif)

In [ ]:
# ============================================
# HYPERPARAMETRES
# ============================================
CONFIG = {
    "epochs": 30,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.1,
    "patience": 7,
    "phase1_end": 5,    # Epochs 1-5  : tête seule
    "phase2_end": 15,   # Epochs 6-15 : derniers 2 blocs
                        # Epochs 16+  : tout le réseau
}

# Phase 1 : geler le backbone
print("📌 Phase 1 : Tête uniquement")
geler_backbone(model)

criterion = LabelSmoothingCE(CONFIG["label_smoothing"])
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"]
)
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-6)
scaler = GradScaler()
early_stopping = EarlyStopping(patience=CONFIG["patience"])

# Historique
history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "val_top5": [],
    "lr": []
}
best_val_acc = 0.0

print(f"\n🏋️ Début de l'entraînement ({CONFIG['epochs']} epochs)")
print(f"   LR          : {CONFIG['lr']}")
print(f"   Smoothing   : {CONFIG['label_smoothing']}")
print(f"   Patience    : {CONFIG['patience']}")
print(f"   Phases      : [1-{CONFIG['phase1_end']}] tête, [{CONFIG['phase1_end']+1}-{CONFIG['phase2_end']}] derniers blocs, [{CONFIG['phase2_end']+1}+] tout")
print("=" * 100)

total_start = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    epoch_start = time.time()
    
    # === Unfreeze progressif ===
    if epoch == CONFIG["phase1_end"] + 1:
        print(f"\n📌 Phase 2 : Derniers 2 blocs")
        degeler_derniers_blocs(model, nb_blocs=2)
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CONFIG["lr"] * 0.5, weight_decay=CONFIG["weight_decay"]
        )
    elif epoch == CONFIG["phase2_end"] + 1:
        print(f"\n📌 Phase 3 : Réseau complet")
        degeler_tout(model)
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CONFIG["lr"] * 0.1, weight_decay=CONFIG["weight_decay"]
        )
    
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
    
    # Validate
    val_loss, val_top1, val_top5 = validate(model, val_loader, criterion)
    
    scheduler.step()
    lr = optimizer.param_groups[0]["lr"]
    elapsed = time.time() - epoch_start
    
    # Historique
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_top1)
    history["val_top5"].append(val_top5)
    history["lr"].append(lr)
    
    # Meilleur modèle
    is_best = val_top1 > best_val_acc
    if is_best:
        best_val_acc = val_top1
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_accuracy": val_top1,
            "val_loss": val_loss,
            "config": CONFIG,
            "class_to_idx": class_mapping,
            "num_classes": NUM_CLASSES,
        }, MODELS_DIR / "efficientnet_b4_best.pth")
    
    # Log
    marker = " 🏆 BEST" if is_best else ""
    print(
        f"Epoch {epoch:2d}/{CONFIG['epochs']} | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:5.1f}% | "
        f"Val Loss: {val_loss:.4f} Top1: {val_top1*100:5.1f}% Top5: {val_top5*100:5.1f}% | "
        f"LR: {lr:.6f} | {elapsed:.0f}s{marker}"
    )
    
    # Early stopping
    if early_stopping(val_loss):
        print(f"\n⏹️ Arrêt anticipé à l'epoch {epoch}")
        break

total_time = time.time() - total_start
print("\n" + "=" * 100)
print(f"🏆 MEILLEURE ACCURACY : {best_val_acc*100:.2f}%")
print(f"⏱️  Temps total : {total_time/60:.1f} minutes")
print(f"💾 Modèle sauvé : {MODELS_DIR / 'efficientnet_b4_best.pth'}")

## 8️⃣ Courbes d'entraînement

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train', markersize=4)
axes[0].plot(epochs_range, history['val_loss'], 'r-o', label='Val', markersize=4)
axes[0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', label='Train', markersize=4)
axes[1].plot(epochs_range, [a*100 for a in history['val_acc']], 'r-o', label='Val Top-1', markersize=4)
axes[1].plot(epochs_range, [a*100 for a in history['val_top5']], 'g-s', label='Val Top-5', markersize=4)
axes[1].set_title('Accuracy (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning Rate
axes[2].plot(epochs_range, history['lr'], 'g-o', markersize=4)
axes[2].set_title('Learning Rate', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

# Lignes de phases
for ax in axes:
    ax.axvline(x=CONFIG['phase1_end']+0.5, color='orange', linestyle='--', alpha=0.5, label='Phase 2')
    ax.axvline(x=CONFIG['phase2_end']+0.5, color='purple', linestyle='--', alpha=0.5, label='Phase 3')

plt.suptitle(f'EfficientNet-B4 — {NUM_CLASSES} classes — Best: {best_val_acc*100:.1f}%', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(str(MODELS_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 Courbes sauvées : {MODELS_DIR / 'training_curves.png'}")

## 9️⃣ Évaluation sur le Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Charger le meilleur modèle
print("📥 Chargement du meilleur modèle...")
checkpoint = torch.load(MODELS_DIR / "efficientnet_b4_best.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"   Epoch {checkpoint['epoch']} — Val Accuracy: {checkpoint['val_accuracy']*100:.2f}%")

# Prédictions sur le test set
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Test"):
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Métriques
test_acc = (all_preds == all_labels).mean()
top5_acc = top_k_accuracy_score(all_labels, all_probs, k=min(5, NUM_CLASSES))

print(f"\n{'='*60}")
print(f"🏆 RÉSULTATS SUR LE TEST SET")
print(f"{'='*60}")
print(f"   Top-1 Accuracy : {test_acc*100:.2f}%")
print(f"   Top-5 Accuracy : {top5_acc*100:.2f}%")
print(f"   Images test    : {len(all_labels)}")
print(f"{'='*60}")

In [ ]:
# Classification report (top 20 classes)
idx_to_class = {v: k for k, v in class_mapping.items()}
present_labels = sorted(set(all_labels.tolist()))
target_names = [idx_to_class.get(i, f"class_{i}") for i in present_labels]

report = classification_report(
    all_labels, all_preds,
    labels=present_labels,
    target_names=target_names,
    output_dict=True,
    zero_division=0
)

# Trier par f1-score
class_results = []
for cls_name in target_names:
    if cls_name in report:
        r = report[cls_name]
        class_results.append((cls_name, r['precision'], r['recall'], r['f1-score'], int(r['support'])))

class_results.sort(key=lambda x: x[3], reverse=True)

print(f"\n📊 Top 20 classes (par F1-score) :")
print(f"{'Classe':<25s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Support':>8s}")
print("-" * 68)
for name, p, r, f1, s in class_results[:20]:
    print(f"{name:<25s} {p*100:>9.1f}% {r*100:>9.1f}% {f1*100:>9.1f}% {s:>7d}")

print(f"\n   Macro avg F1 : {report['macro avg']['f1-score']*100:.2f}%")
print(f"   Weighted avg F1 : {report['weighted avg']['f1-score']*100:.2f}%")

In [ ]:
# Matrice de confusion (top 20 classes les plus fréquentes)
top20_labels = present_labels[:20]
top20_names = [idx_to_class.get(i, f"c{i}") for i in top20_labels]

mask = np.isin(all_labels, top20_labels)
if mask.sum() > 0:
    cm = confusion_matrix(all_labels[mask], all_preds[mask], labels=top20_labels)
    
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=top20_names, yticklabels=top20_names, ax=ax)
    ax.set_xlabel('Prédit', fontsize=12)
    ax.set_ylabel('Vrai', fontsize=12)
    ax.set_title(f'Matrice de Confusion (Top 20 classes) — Accuracy: {test_acc*100:.1f}%',
                 fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(str(MODELS_DIR / 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Matrice sauvée : {MODELS_DIR / 'confusion_matrix.png'}")
else:
    print("⚠️  Pas assez de données pour la matrice de confusion")

## 💾 Extraction des embeddings visuels (pour la recherche par image)

In [ ]:
import pickle

# ============================================
# Extraire les embeddings avec le backbone
# (features avant la couche classifier)
# ============================================
print("🔍 Extraction des embeddings visuels...")

# Créer un extracteur de features
feature_extractor = timm.create_model(
    "efficientnet_b4",
    pretrained=False,
    num_classes=0  # Supprime la tête → retourne les features
)

# Charger les poids du backbone depuis notre modèle entraîné
state_dict = checkpoint["model_state_dict"]
# Filtrer les clés du classifier
backbone_dict = {k: v for k, v in state_dict.items() if not k.startswith("classifier")}
feature_extractor.load_state_dict(backbone_dict, strict=False)
feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()

# Extraire les features de TOUTES les images
all_embeddings = []
all_embed_labels = []
all_embed_paths = []

# Parcourir le dossier raw/ (toutes les 720 images)
embed_transform = val_transform

for cat_dir in sorted(RAW_DIR.iterdir()):
    if not cat_dir.is_dir():
        continue
    cat_name = cat_dir.name
    for img_path in sorted(cat_dir.glob("*.jpg")):
        try:
            img = Image.open(img_path).convert("RGB")
            img_tensor = embed_transform(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                feat = feature_extractor(img_tensor)
            all_embeddings.append(feat.cpu().numpy().flatten())
            all_embed_labels.append(cat_name)
            all_embed_paths.append(str(img_path.name))
        except Exception as e:
            print(f"   ⚠️ Erreur {img_path.name}: {e}")

embeddings_array = np.array(all_embeddings)
print(f"\n✅ {len(all_embeddings)} embeddings extraits")
print(f"   Shape : {embeddings_array.shape}")
print(f"   Dimension : {embeddings_array.shape[1]}")

# Sauvegarder
embeddings_data = {
    "embeddings": embeddings_array,
    "labels": all_embed_labels,
    "paths": all_embed_paths,
    "embedding_dim": embeddings_array.shape[1],
}
with open(MODELS_DIR / "product_embeddings.pkl", "wb") as f:
    pickle.dump(embeddings_data, f)

print(f"💾 Embeddings sauvés : {MODELS_DIR / 'product_embeddings.pkl'}")

## 💾 Sauvegarder l'historique & télécharger les fichiers

In [ ]:
# Sauvegarder l'historique
history_json = {
    k: [float(v) for v in vals] for k, vals in history.items()
}
history_json["best_val_accuracy"] = float(best_val_acc)
history_json["test_accuracy"] = float(test_acc)
history_json["test_top5_accuracy"] = float(top5_acc)
history_json["config"] = CONFIG
history_json["num_classes"] = NUM_CLASSES
history_json["total_train_images"] = len(train_dataset)

with open(MODELS_DIR / "training_history.json", "w") as f:
    json.dump(history_json, f, indent=2)

# Sauvegarder le class_mapping dans models/
with open(MODELS_DIR / "class_mapping.json", "w") as f:
    json.dump(class_mapping, f, indent=2)

print("📦 Fichiers générés :")
for f in sorted(MODELS_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"   {f.name} ({size_mb:.1f} Mo)")

In [ ]:
# === TÉLÉCHARGER TOUS LES FICHIERS ===
import shutil

# Créer une archive ZIP de tout
zip_path = "/content/ECommerce_IA_models"
shutil.make_archive(zip_path, 'zip', MODELS_DIR)

print(f"\n📥 Archive créée : {zip_path}.zip")
zip_size = os.path.getsize(f"{zip_path}.zip") / 1e6
print(f"   Taille : {zip_size:.1f} Mo")

# Télécharger automatiquement
try:
    from google.colab import files
    print("\n⬇️  Téléchargement automatique...")
    files.download(f"{zip_path}.zip")
    print("✅ Téléchargement lancé ! Vérifiez vos téléchargements.")
except ImportError:
    print("\n⚠️  Pas sur Colab — fichiers disponibles dans /content/models/")
    print("   Téléchargez manuellement via le panneau fichiers (à gauche)")

## ✅ Instructions après téléchargement

1. **Dézippez** `ECommerce_IA_models.zip` dans votre projet local
2. **Copiez les fichiers** :
   ```
   efficientnet_b4_best.pth  →  models/classification/
   product_embeddings.pkl    →  models/classification/
   class_mapping.json        →  models/classification/
   training_history.json     →  models/classification/
   training_curves.png       →  results/
   confusion_matrix.png      →  results/
   ```
3. **Lancez l'API** : `python -m api.main`
4. **Testez** : `curl -X POST -F 'file=@photo.jpg' http://localhost:8000/classify`

---

🎉 **Félicitations !** Le modèle est entraîné et prêt pour la production.